# Check 04 — Tenant and Limits

In [ ]:
import pathlib
import sys
import tempfile

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
_contracts_src = _root / "packages" / "eXo_adapters" / "packages" / "exo-brain-core-contracts" / "src"
if _contracts_src.is_dir():
    sys.path.insert(0, str(_contracts_src))

from src.tenancy.quotas import TenantQuotaManager
from src.tenancy.rate_limiter import TenantRateLimiter, SQLiteTenantRateLimiter

In [ ]:
quota = TenantQuotaManager(max_active_jobs_per_tenant=2, hard_enforcement=True)
assert quota.check_submission("tenant-a", active_jobs=0).allowed
assert quota.check_submission("tenant-a", active_jobs=1).allowed
denied = quota.check_submission("tenant-a", active_jobs=2)
assert not denied.allowed
print("quota denial:", denied.reason_code)

mem_limiter = TenantRateLimiter(max_requests=2, window_seconds=60)
a1, _ = mem_limiter.allow("tenant-a")
a2, _ = mem_limiter.allow("tenant-a")
a3, retry = mem_limiter.allow("tenant-a")
assert a1 and a2 and (not a3) and retry > 0
print("memory limiter ok")

with tempfile.TemporaryDirectory() as tmp:
    db_path = str(pathlib.Path(tmp) / "limits.db")
    sqlite_limiter = SQLiteTenantRateLimiter(
        db_path=db_path, max_requests=1, window_seconds=60, limiter_id="turns"
    )
    ok1, _ = sqlite_limiter.allow("tenant-b")
    ok2, retry2 = sqlite_limiter.allow("tenant-b")
    assert ok1 and (not ok2) and retry2 > 0
    print("sqlite limiter ok")

print("PASS: tenancy and limits checks")